<a href="https://colab.research.google.com/github/Harshada900/pyspark-practise/blob/main/9_Handling_Nulls_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pyspark.sql.functions as f
from pyspark.sql import SparkSession
from pyspark.sql.types import *


In [ ]:
spark = SparkSession.builder.appName("Session8").config("spark.sql.shuffle.partitions", "5").getOrCreate()

In [ ]:
data = [(1, "Raj",   "India",  50000.0, 25),
        (2, "Priya", "USA",    None,    28),
        (3, "Amit",  None,     60000.0, None),
        (4, "Sara",  "UK",     80000.0, 32),
        (5, "Karan", None,     None,    27),
        (6, None,    "India",  55000.0, None)]

columns = ["emp_id", "name", "country", "salary", "age"]
df = spark.createDataFrame(data, columns)
df.show()

+------+-----+-------+-------+----+
|emp_id| name|country| salary| age|
+------+-----+-------+-------+----+
|     1|  Raj|  India|50000.0|  25|
|     2|Priya|    USA|   NULL|  28|
|     3| Amit|   NULL|60000.0|NULL|
|     4| Sara|     UK|80000.0|  32|
|     5|Karan|   NULL|   NULL|  27|
|     6| NULL|  India|55000.0|NULL|
+------+-----+-------+-------+----+



## **1. Detecting Nulls — isNull / isNotNull**

In [ ]:
df.filter(f.col("salary").isNull()).show()    #filtering rows where salary is null

+------+-----+-------+------+---+
|emp_id| name|country|salary|age|
+------+-----+-------+------+---+
|     2|Priya|    USA|  NULL| 28|
|     5|Karan|   NULL|  NULL| 27|
+------+-----+-------+------+---+



In [ ]:
df.filter(f.col("salary").isNotNull()).show()   #filtering rows where salary is not null

+------+----+-------+-------+----+
|emp_id|name|country| salary| age|
+------+----+-------+-------+----+
|     1| Raj|  India|50000.0|  25|
|     3|Amit|   NULL|60000.0|NULL|
|     4|Sara|     UK|80000.0|  32|
|     6|NULL|  India|55000.0|NULL|
+------+----+-------+-------+----+



## **Count Nulls Per Column — Real World Usage**

In [ ]:
df.select([ f.sum(f.when(f.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]).show()

+------+----+-------+------+---+
|emp_id|name|country|salary|age|
+------+----+-------+------+---+
|     0|   1|      2|     2|  2|
+------+----+-------+------+---+



### **2. dropna — Drop Rows with Nulls**

In [ ]:
# Drop rows where ANY column has null

df.dropna().show()

+------+----+-------+-------+---+
|emp_id|name|country| salary|age|
+------+----+-------+-------+---+
|     1| Raj|  India|50000.0| 25|
|     4|Sara|     UK|80000.0| 32|
+------+----+-------+-------+---+



In [ ]:
# Drop rows where ALL columns are null (rarely needed)
df.dropna(how="all").show()

+------+-----+-------+-------+----+
|emp_id| name|country| salary| age|
+------+-----+-------+-------+----+
|     1|  Raj|  India|50000.0|  25|
|     2|Priya|    USA|   NULL|  28|
|     3| Amit|   NULL|60000.0|NULL|
|     4| Sara|     UK|80000.0|  32|
|     5|Karan|   NULL|   NULL|  27|
|     6| NULL|  India|55000.0|NULL|
+------+-----+-------+-------+----+



In [ ]:
# Drop rows where specific columns at least one null

df.dropna(subset=["salary", "age"], how="any").show()

+------+----+-------+-------+---+
|emp_id|name|country| salary|age|
+------+----+-------+-------+---+
|     1| Raj|  India|50000.0| 25|
|     4|Sara|     UK|80000.0| 32|
+------+----+-------+-------+---+



In [ ]:
#drop rows if all specific columns has nulls

df.dropna(subset=["salary", "age"], how="all").show()

df.dropna(subset=["salary", "country"], how="all").show()

+------+-----+-------+-------+----+
|emp_id| name|country| salary| age|
+------+-----+-------+-------+----+
|     1|  Raj|  India|50000.0|  25|
|     2|Priya|    USA|   NULL|  28|
|     3| Amit|   NULL|60000.0|NULL|
|     4| Sara|     UK|80000.0|  32|
|     5|Karan|   NULL|   NULL|  27|
|     6| NULL|  India|55000.0|NULL|
+------+-----+-------+-------+----+

+------+-----+-------+-------+----+
|emp_id| name|country| salary| age|
+------+-----+-------+-------+----+
|     1|  Raj|  India|50000.0|  25|
|     2|Priya|    USA|   NULL|  28|
|     3| Amit|   NULL|60000.0|NULL|
|     4| Sara|     UK|80000.0|  32|
|     6| NULL|  India|55000.0|NULL|
+------+-----+-------+-------+----+



## **3. fillna — Replace Nulls with a Value**

In [ ]:
# Replace ALL nulls in entire DataFrame with a value

df.fillna(0).show()   #string col remains null while numeric cols gets replace with null

+------+-----+-------+-------+---+
|emp_id| name|country| salary|age|
+------+-----+-------+-------+---+
|     1|  Raj|  India|50000.0| 25|
|     2|Priya|    USA|    0.0| 28|
|     3| Amit|   NULL|60000.0|  0|
|     4| Sara|     UK|80000.0| 32|
|     5|Karan|   NULL|    0.0| 27|
|     6| NULL|  India|55000.0|  0|
+------+-----+-------+-------+---+



In [ ]:
# Replace nulls in specific columns
df.fillna(0, subset=["salary", "age"]).fillna("Unknown", subset=["country"]).show()


+------+-----+-------+-------+---+
|emp_id| name|country| salary|age|
+------+-----+-------+-------+---+
|     1|  Raj|  India|50000.0| 25|
|     2|Priya|    USA|    0.0| 28|
|     3| Amit|Unknown|60000.0|  0|
|     4| Sara|     UK|80000.0| 32|
|     5|Karan|Unknown|    0.0| 27|
|     6| NULL|  India|55000.0|  0|
+------+-----+-------+-------+---+



In [ ]:
# Replace nulls with different values per column

df.fillna(
    {
        "name": "Unknown",
        "country": "Unknown",
        "salary": 0.0,
        "age": 0
    }
).show()

+------+-------+-------+-------+---+
|emp_id|   name|country| salary|age|
+------+-------+-------+-------+---+
|     1|    Raj|  India|50000.0| 25|
|     2|  Priya|    USA|    0.0| 28|
|     3|   Amit|Unknown|60000.0|  0|
|     4|   Sara|     UK|80000.0| 32|
|     5|  Karan|Unknown|    0.0| 27|
|     6|Unknown|  India|55000.0|  0|
+------+-------+-------+-------+---+



## **4. fill Nulls with Average/Mean — Real World Pattern**

In [ ]:
#called as mean imputation
mean_salary = df.select(f.avg("salary")).collect()[0][0]

df.fillna({"salary":mean_salary}).show()

+------+-----+-------+-------+----+
|emp_id| name|country| salary| age|
+------+-----+-------+-------+----+
|     1|  Raj|  India|50000.0|  25|
|     2|Priya|    USA|61250.0|  28|
|     3| Amit|   NULL|60000.0|NULL|
|     4| Sara|     UK|80000.0|  32|
|     5|Karan|   NULL|61250.0|  27|
|     6| NULL|  India|55000.0|NULL|
+------+-----+-------+-------+----+



## **5. when().otherwise() for Complex Null Handling**

In [ ]:
df.withColumn("salary", f.when(f.col("salary").isNull() & (f.col("country")=="USA"), 70000.0)
              .when(f.col("salary").isNull() & (f.col("country")=="India"), 50000.0)
              .otherwise(f.col("salary"))).show()

+------+-----+-------+-------+----+
|emp_id| name|country| salary| age|
+------+-----+-------+-------+----+
|     1|  Raj|  India|50000.0|  25|
|     2|Priya|    USA|70000.0|  28|
|     3| Amit|   NULL|60000.0|NULL|
|     4| Sara|     UK|80000.0|  32|
|     5|Karan|   NULL|   NULL|  27|
|     6| NULL|  India|55000.0|NULL|
+------+-----+-------+-------+----+



🔨 Practice Task

Using our DataFrame:

Count nulls in every column

Fill null salary with the mean salary

Fill null country with "Unknown"

Fill null age with 0

Drop rows where name is null

Do all of this in a clean pipeline — chained together!

In [ ]:
df.select([f.sum(f.when(f.col(c).isNull(), 1).otherwise(0)).alias(c)
 for (c) in df.columns

           ]).show()

+------+----+-------+------+---+
|emp_id|name|country|salary|age|
+------+----+-------+------+---+
|     0|   1|      2|     2|  2|
+------+----+-------+------+---+



In [ ]:
mean_salary = df.select(f.avg(f.col("salary"))).collect()[0][0]

In [ ]:
df.fillna({"salary": mean_salary}).show()

+------+-----+-------+-------+----+
|emp_id| name|country| salary| age|
+------+-----+-------+-------+----+
|     1|  Raj|  India|50000.0|  25|
|     2|Priya|    USA|61250.0|  28|
|     3| Amit|   NULL|60000.0|NULL|
|     4| Sara|     UK|80000.0|  32|
|     5|Karan|   NULL|61250.0|  27|
|     6| NULL|  India|55000.0|NULL|
+------+-----+-------+-------+----+



In [ ]:
df.fillna({"country":"Unknown", "age": 0}).show()

+------+-----+-------+-------+---+
|emp_id| name|country| salary|age|
+------+-----+-------+-------+---+
|     1|  Raj|  India|50000.0| 25|
|     2|Priya|    USA|   NULL| 28|
|     3| Amit|Unknown|60000.0|  0|
|     4| Sara|     UK|80000.0| 32|
|     5|Karan|Unknown|   NULL| 27|
|     6| NULL|  India|55000.0|  0|
+------+-----+-------+-------+---+



In [ ]:
df.dropna(subset=["name"]).show()

+------+-----+-------+-------+----+
|emp_id| name|country| salary| age|
+------+-----+-------+-------+----+
|     1|  Raj|  India|50000.0|  25|
|     2|Priya|    USA|   NULL|  28|
|     3| Amit|   NULL|60000.0|NULL|
|     4| Sara|     UK|80000.0|  32|
|     5|Karan|   NULL|   NULL|  27|
+------+-----+-------+-------+----+



In [ ]:
# all above in one clean pipeline

df.select([f.sum(f.when(f.col(c).isNull(), 1).otherwise(0)).alias(c)
 for (c) in df.columns])

DataFrame[emp_id: bigint, name: bigint, country: bigint, salary: bigint, age: bigint]

In [ ]:
df2 = df.fillna({"country":"Unknown", "age": 0, "salary":mean_salary}).dropna(subset=["name"])
df2.show()

+------+-----+-------+-------+---+
|emp_id| name|country| salary|age|
+------+-----+-------+-------+---+
|     1|  Raj|  India|50000.0| 25|
|     2|Priya|    USA|61250.0| 28|
|     3| Amit|Unknown|60000.0|  0|
|     4| Sara|     UK|80000.0| 32|
|     5|Karan|Unknown|61250.0| 27|
+------+-----+-------+-------+---+



In [ ]:
df.show()

+------+-----+-------+-------+----+
|emp_id| name|country| salary| age|
+------+-----+-------+-------+----+
|     1|  Raj|  India|50000.0|  25|
|     2|Priya|    USA|   NULL|  28|
|     3| Amit|   NULL|60000.0|NULL|
|     4| Sara|     UK|80000.0|  32|
|     5|Karan|   NULL|   NULL|  27|
|     6| NULL|  India|55000.0|NULL|
+------+-----+-------+-------+----+

